# Bibliotecas

In [115]:
import os
import onnx
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader,Dataset
from torchvision.models import resnet18, ResNet18_Weights
from brevitas.nn import QuantLinear, QuantReLU, QuantConv2d,QuantIdentity,TruncAvgPool2d
from brevitas.export import export_qonnx
import copy
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, cohen_kappa_score,recall_score
import matplotlib.pyplot as plt
import seaborn as sns
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.datatype import DataType
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.fold_constants import FoldConstants
from finn.util.visualization import showInNetron
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import shutil
from finn.transformation.streamline import Streamline
from brevitas.quant import Int8ActPerTensorFloat
from brevitas.quant import Int8WeightPerTensorFloat,Int8WeightPerChannelFloat
from brevitas.quant import Uint8ActPerTensorFloat
from brevitas.quant import TruncTo8bit
from brevitas.core.restrict_val import RestrictValueType
import warnings
from finn.builder.build_dataflow_config import (
    DataflowBuildConfig,
    ShellFlowType,
)
from qonnx.transformation.general import (
    ConvertDivToMul,
    ConvertSubToAdd
)
from finn.transformation.streamline.sign_to_thres import ConvertSignToThres

from finn.transformation.streamline.absorb import (
    AbsorbAddIntoMultiThreshold,
    AbsorbMulIntoMultiThreshold,
    FactorOutMulSignMagnitude,
    Absorb1BitMulIntoConv,
    AbsorbScalarMulAddIntoTopK,
    AbsorbTransposeIntoFlatten
)
from finn.transformation.streamline.collapse_repeated import (
    CollapseRepeatedMul,
    CollapseRepeatedAdd
)
from finn.builder.build_dataflow_steps import VerificationStepType, verify_step, step_specialize_layers
from qonnx.transformation.remove import RemoveIdentityOps
from qonnx.transformation.batchnorm_to_affine import BatchNormToAffine

from finn.transformation.streamline.reorder import (
    MoveTransposePastFork,
    MoveTransposePastJoinAdd,
    MoveMaxPoolPastMultiThreshold,
    MoveMulPastMaxPool,
    MoveScalarLinearPastInvariants,
    MoveScalarMulPastConv,
    MoveTransposePastScalarMul,
    MoveAddPastMul,
    MoveAddPastConv,
    MoveMulPastFork,
    MoveIdenticalOpPastJoinOp,
    MoveOpPastFork,
    MoveLinearPastEltwiseAdd,
    MoveScalarMulPastMatMul,
    MoveScalarAddPastMatMul
    

)

from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from qonnx.util.cleanup import cleanup_model
from finn.transformation.streamline.absorb import (AbsorbConsecutiveTransposes, AbsorbSignBiasIntoMultiThreshold, AbsorbTransposeIntoMultiThreshold)
from qonnx.transformation.general import SortGraph
from finn.transformation.fpgadataflow import convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.insert_dwc import InsertDWC
from qonnx.transformation.general import RemoveUnusedTensors
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.double_to_single_float import DoubleToSingleFloat
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from qonnx.transformation.infer_data_layouts import InferDataLayouts

In [2]:
!pip install seaborn

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 KB 2.0 MB/s eta 0:00:00a 0:00:01


# Dataset

In [149]:
BASE_PATH = 'DDR-dataset/DR_grading'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VALID_PATH = os.path.join(BASE_PATH, 'valid')
TEST_PATH = os.path.join(BASE_PATH, 'test')

In [150]:
def load_labels(file_path):
    labels = pd.read_csv(file_path, sep=' ', header=None, names=['filename', 'label'])
    return labels

train_labels = load_labels(os.path.join(BASE_PATH, 'train.txt'))
valid_labels = load_labels(os.path.join(BASE_PATH, 'valid.txt'))
test_labels = load_labels(os.path.join(BASE_PATH, 'test.txt'))

In [151]:
class CustomDataset(Dataset):
    def __init__(self, labels, dir_path, transform=None):
        self.labels = labels
        self.dir_path = dir_path
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_name = os.path.join(self.dir_path, self.labels.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        label = int(self.labels.iloc[idx, 1])
        if self.transform:
            image = self.transform(image)
        return image, label

In [152]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [153]:
BATCH_SIZE = 16  # Tamanho do batch

train_dataset = CustomDataset(train_labels, TRAIN_PATH, transform=train_transform)
valid_dataset = CustomDataset(valid_labels, VALID_PATH, transform=test_transform)
test_dataset = CustomDataset(test_labels, TEST_PATH, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Definindo modelo full precision

In [4]:
model =  resnet18(weights=ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /tmp/home_dir/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


  0%|          | 0.00/44.7M [00:00<?, ?B/s]

In [12]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
num_ftrs = model.fc.in_features #512
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(128, 6)
)
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

# Definindo modelo quantizado

In [5]:
# Copyright (C) 2023, Advanced Micro Devices, Inc. All rights reserved.
# SPDX-License-Identifier: BSD-3-Clause

class CommonIntWeightPerTensorQuant(Int8WeightPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None

class CommonIntWeightPerChannelQuant(CommonIntWeightPerTensorQuant):
    scaling_per_output_channel = True

class CommonIntActQuant(Int8ActPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None
    restrict_scaling_type = RestrictValueType.LOG_FP

class CommonUintActQuant(Uint8ActPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None
    restrict_scaling_type = RestrictValueType.LOG_FP

In [6]:
class QuantBasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, weight_bit_width, act_bit_width, stride=1, downsample=None):
        super(QuantBasicBlock, self).__init__()

        self.conv1 = QuantConv2d(in_channels, out_channels, kernel_size=3, padding=1,
                                 stride=stride,  # <-- AQUI estava faltando
                                 bias=False,
                                 weight_bit_width=weight_bit_width,
                                 weight_quant=CommonIntWeightPerChannelQuant)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu1 = QuantReLU(return_quant_tensor=True, bit_width=act_bit_width)

        self.conv2 = QuantConv2d(out_channels, out_channels, kernel_size=3, padding=1,
                                 bias=False,
                                 weight_bit_width=weight_bit_width,
                                 weight_quant=CommonIntWeightPerChannelQuant)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu2 = QuantReLU(return_quant_tensor=True, bit_width=act_bit_width)

        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        if hasattr(out, 'value'):
            out = out.value
        if hasattr(identity, 'value'):
            identity = identity.value

        out = out + identity
        out = self.relu2(out)

        return out

class QuantResNet18(nn.Module):
    def __init__(self,weight_bit_width, act_bit_width):
        super(QuantResNet18, self).__init__()
        self.in_channels = 64
        self.conv1 = QuantConv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False,
                                weight_bit_width=8, weight_quant=Int8WeightPerChannelFloat)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = QuantReLU(return_quant_tensor=True,bit_width=act_bit_width)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)

        self.layer1 = self._make_layer(64,2,weight_bit_width,act_bit_width)
        self.layer2 = self._make_layer(128,2,weight_bit_width,act_bit_width,stride=2)
        self.layer3 = self._make_layer(256,2,weight_bit_width,act_bit_width,stride=2)
        self.layer4 = self._make_layer(512,2,weight_bit_width,act_bit_width,stride=2)

        self.avgpool = TruncAvgPool2d(kernel_size=4,trunc_quant=TruncTo8bit,float_to_int_impl_type='FLOOR')   
        self.fc = nn.Sequential(
            QuantLinear(512, 256, bias=True,
                       weight_bit_width=weight_bit_width,weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(return_quant_tensor=True,act_bit_width=act_bit_width),
            nn.Dropout(0.4),
            QuantLinear(256, 128,bias=True,
                       weight_bit_width=weight_bit_width,weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(return_quant_tensor=True,act_bit_width=act_bit_width),
            nn.Dropout(0.4),
            QuantLinear(128, 6, bias=True,
                        weight_bit_width=8,weight_quant=Int8WeightPerTensorFloat)
        )
        #inicializando pesos
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self,out_channels,blocks,weight_bit_width, act_bit_width,stride=1):
            downsample = None
            if stride != 1 or self.in_channels != out_channels:
                downsample = nn.Sequential(
                    QuantConv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False,
                               weight_bit_width=weight_bit_width, weight_quant=CommonIntWeightPerChannelQuant),
                    nn.BatchNorm2d(out_channels)
                )
            layers = [QuantBasicBlock(self.in_channels, out_channels, weight_bit_width,act_bit_width, stride, downsample)]
            self.in_channels = out_channels
            for _ in range(1,blocks):
                layers.append(QuantBasicBlock(out_channels, out_channels, weight_bit_width, act_bit_width))
            return nn.Sequential(*layers)
            
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x,1)
        x = self.fc(x)
        return x

In [43]:
quant_model = QuantResNet18(8,8)
quant_model

QuantResNet18(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

# Treinando modelo não quantizado

In [33]:
def train_model(model, criterion, optimizer, train_loader, valid_loader, model_name, num_epochs=20):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                data_loader = train_loader
            else:
                model.eval()
                data_loader = valid_loader
                
            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []
            
            for inputs, labels in data_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            epoch_loss = running_loss / len(data_loader.dataset)
            epoch_acc = running_corrects.double() / len(data_loader.dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(best_model_wts, model_name)
    
    print(f'Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

In [34]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [38]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)  

In [42]:
model = train_model(model, criterion, optimizer, train_loader, valid_loader,"best_resnet18.pth", num_epochs=20)

Epoch 1/20
----------
train Loss: 1.1154 Acc: 0.5707
val Loss: 0.8016 Acc: 0.7578
Epoch 2/20
----------
train Loss: 0.9241 Acc: 0.6628
val Loss: 0.7451 Acc: 0.7373
Epoch 3/20
----------
train Loss: 0.8619 Acc: 0.6792
val Loss: 0.6493 Acc: 0.7856
Epoch 4/20
----------
train Loss: 0.8261 Acc: 0.6977
val Loss: 0.7170 Acc: 0.7629
Epoch 5/20
----------
train Loss: 0.7868 Acc: 0.7113
val Loss: 0.7448 Acc: 0.7468
Epoch 6/20
----------
train Loss: 0.7676 Acc: 0.7220
val Loss: 0.6143 Acc: 0.8020
Epoch 7/20
----------
train Loss: 0.7462 Acc: 0.7248
val Loss: 0.6082 Acc: 0.8185
Epoch 8/20
----------
train Loss: 0.7167 Acc: 0.7400
val Loss: 0.6315 Acc: 0.7878
Epoch 9/20
----------
train Loss: 0.7175 Acc: 0.7403
val Loss: 0.5915 Acc: 0.8094
Epoch 10/20
----------
train Loss: 0.6916 Acc: 0.7453
val Loss: 0.5722 Acc: 0.8225
Epoch 11/20
----------
train Loss: 0.6737 Acc: 0.7604
val Loss: 0.7341 Acc: 0.7523
Epoch 12/20
----------
train Loss: 0.6631 Acc: 0.7574
val Loss: 0.5770 Acc: 0.8053
Epoch 13/20
-

# Avaliando modelo não quantizado

In [44]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("best_resnet18.pth", map_location=device))
model = model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [45]:
def calculate_metrics(loader, model, device):
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)], output_dict=True)
    kappa = cohen_kappa_score(all_labels, all_preds)
    
    per_class_accuracy = [report[str(i)]['precision'] for i in range(6)]
    mean_accuracy = sum(per_class_accuracy) / len(per_class_accuracy)

    return accuracy, mean_accuracy, kappa, classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)]), confusion_matrix(all_labels, all_preds)

In [47]:
def plot_confusion_matrix(cm, title, filename):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=[str(i) for i in range(6)], yticklabels=[str(i) for i in range(6)])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.savefig(filename)
    plt.close()

In [55]:
print("Validation Set Metrics:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report:")
print(valid_report)
print("Validation Confusion Matrix:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet18.png')

Validation Set Metrics:
Validation OA: 0.8339
Validation AA: 0.6795
Validation Kappa: 0.7443
Validation Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1253
           1       0.35      0.05      0.08       126
           2       0.81      0.81      0.81       895
           3       0.34      0.53      0.41        47
           4       0.74      0.69      0.72       182
           5       0.96      0.80      0.87       230

    accuracy                           0.83      2733
   macro avg       0.68      0.64      0.64      2733
weighted avg       0.82      0.83      0.82      2733

Validation Confusion Matrix:
[[1211    5   35    0    2    0]
 [  68    6   49    0    3    0]
 [  99    6  726   42   18    4]
 [   0    0   16   25    6    0]
 [   3    0   42    7  126    4]
 [   5    0   25    0   15  185]]


In [54]:
print("Test Set Metrics:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report:")
print(test_report)
print("Test Confusion Matrix:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet18.png')

Test Set Metrics:
Test OA: 0.7235
Test AA: 0.6002
Test Kappa: 0.5637
Test Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.97      0.82      1880
           1       0.16      0.02      0.03       189
           2       0.72      0.47      0.57      1344
           3       0.39      0.31      0.35        71
           4       0.80      0.60      0.69       275
           5       0.81      0.93      0.87       346

    accuracy                           0.72      4105
   macro avg       0.60      0.55      0.55      4105
weighted avg       0.70      0.72      0.69      4105

Test Confusion Matrix:
[[1826    4   50    0    0    0]
 [ 115    3   68    0    1    2]
 [ 604   12  631   24   23   50]
 [   2    0   43   22    4    0]
 [   4    0   71   10  165   25]
 [   1    0   10    0   12  323]]


# Treinando modelos quantizados

In [17]:
model.load_state_dict(torch.load("best_resnet18.pth"))

<All keys matched successfully>

In [18]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [13]:
def train_qat(model, criterion, optimizer, train_loader, valid_loader, model_name, num_epochs=10):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                data_loader = train_loader
            else:
                model.eval()
                data_loader = valid_loader
                
            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []
            
            for inputs, labels in data_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            epoch_loss = running_loss / len(data_loader.dataset)
            epoch_acc = running_corrects.double() / len(data_loader.dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            torch.save(model.state_dict(), f'{model_name}_epoch{epoch+1}.pth')
            
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(best_model_wts, model_name)
    
    print(f'Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

In [14]:
BATCH_SIZE = 8  # Tamanho do batch reduzido para caber na GPU

train_dataset = CustomDataset(train_labels, TRAIN_PATH, transform=train_transform)
valid_dataset = CustomDataset(valid_labels, VALID_PATH, transform=test_transform)
test_dataset = CustomDataset(test_labels, TEST_PATH, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [75]:
print("QAT para pesos de 8 bits e ativações de 8 bits.")
model_name = 'qat_resnet18_w8_a8.pth'

quant_model = QuantResNet18(8,8)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 8 bits e ativações de 8 bits.
Epoch 1/10
----------


/tmp/ipykernel_188/3438579381.py:107: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x,1)


train Loss: 0.7139 Acc: 0.7413
val Loss: 0.5826 Acc: 0.8203
Epoch 2/10
----------
train Loss: 0.6886 Acc: 0.7481
val Loss: 0.6389 Acc: 0.7999
Epoch 3/10
----------
train Loss: 0.6688 Acc: 0.7563
val Loss: 0.6575 Acc: 0.7816
Epoch 4/10
----------
train Loss: 0.6512 Acc: 0.7715
val Loss: 0.6145 Acc: 0.7940
Epoch 5/10
----------
train Loss: 0.6339 Acc: 0.7750
val Loss: 0.7000 Acc: 0.7534
Epoch 6/10
----------
train Loss: 0.6047 Acc: 0.7808
val Loss: 0.6866 Acc: 0.7746
Epoch 7/10
----------
train Loss: 0.6015 Acc: 0.7744
val Loss: 0.6371 Acc: 0.7922
Epoch 8/10
----------
train Loss: 0.5895 Acc: 0.7871
val Loss: 0.7248 Acc: 0.7512
Epoch 9/10
----------
train Loss: 0.5768 Acc: 0.7849
val Loss: 0.6130 Acc: 0.8024
Epoch 10/10
----------
train Loss: 0.5728 Acc: 0.7915
val Loss: 0.6695 Acc: 0.7753
Best val Acc: 0.820344


In [76]:
print("QAT para pesos de 8 bits e ativações de 4 bits.")
model_name = 'qat_resnet18_w8_a4.pth'

quant_model = QuantResNet18(8,4)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 8 bits e ativações de 4 bits.
Epoch 1/10
----------


/tmp/ipykernel_188/3438579381.py:107: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x,1)


train Loss: 1.3152 Acc: 0.6645
val Loss: 0.6972 Acc: 0.7801
Epoch 2/10
----------
train Loss: 0.8329 Acc: 0.6989
val Loss: 0.6586 Acc: 0.7801
Epoch 3/10
----------
train Loss: 0.8005 Acc: 0.7062
val Loss: 0.6957 Acc: 0.7757
Epoch 4/10
----------
train Loss: 0.7894 Acc: 0.7140
val Loss: 0.6935 Acc: 0.7636
Epoch 5/10
----------
train Loss: 0.7773 Acc: 0.7127
val Loss: 0.7459 Acc: 0.7494
Epoch 6/10
----------
train Loss: 0.7686 Acc: 0.7160
val Loss: 0.7109 Acc: 0.7805
Epoch 7/10
----------
train Loss: 0.7521 Acc: 0.7261
val Loss: 0.8001 Acc: 0.7131
Epoch 8/10
----------
train Loss: 0.7540 Acc: 0.7213
val Loss: 0.8198 Acc: 0.7205
Epoch 9/10
----------
train Loss: 0.7533 Acc: 0.7244
val Loss: 0.6885 Acc: 0.7805
Epoch 10/10
----------
train Loss: 0.7352 Acc: 0.7346
val Loss: 0.7541 Acc: 0.7303
Best val Acc: 0.780461


In [19]:
print("QAT para pesos de 4 bits e ativações de 8 bits.")
model_name = 'qat_resnet18_w4_a8.pth'

quant_model = QuantResNet18(4,8)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 4 bits e ativações de 8 bits.
Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_189/3438579381.py:107: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x,1)


train Loss: 0.7269 Acc: 0.7397
val Loss: 0.7179 Acc: 0.7589
Epoch 2/10
----------
train Loss: 0.6914 Acc: 0.7485
val Loss: 0.6119 Acc: 0.8083
Epoch 3/10
----------
train Loss: 0.6693 Acc: 0.7548
val Loss: 0.6940 Acc: 0.7797
Epoch 4/10
----------
train Loss: 0.6595 Acc: 0.7627
val Loss: 0.6832 Acc: 0.7794
Epoch 5/10
----------
train Loss: 0.6403 Acc: 0.7687
val Loss: 0.6173 Acc: 0.7980
Epoch 6/10
----------
train Loss: 0.6265 Acc: 0.7726
val Loss: 0.7109 Acc: 0.7757
Epoch 7/10
----------
train Loss: 0.6219 Acc: 0.7767
val Loss: 0.6616 Acc: 0.7845
Epoch 8/10
----------
train Loss: 0.6170 Acc: 0.7756
val Loss: 0.6366 Acc: 0.8039
Epoch 9/10
----------
train Loss: 0.5959 Acc: 0.7801
val Loss: 0.5745 Acc: 0.8145
Epoch 10/10
----------
train Loss: 0.5799 Acc: 0.7854
val Loss: 0.6856 Acc: 0.7556
Best val Acc: 0.814490


In [20]:
print("QAT para pesos de 4 bits e ativações de 4 bits.")
model_name = 'qat_resnet18_w4_a4.pth'

quant_model = QuantResNet18(4,4)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 4 bits e ativações de 4 bits.
Epoch 1/10
----------


/tmp/ipykernel_189/3438579381.py:107: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x,1)


train Loss: 1.3383 Acc: 0.6519
val Loss: 0.7232 Acc: 0.7508
Epoch 2/10
----------
train Loss: 0.8346 Acc: 0.6914
val Loss: 0.7040 Acc: 0.7951
Epoch 3/10
----------
train Loss: 0.8066 Acc: 0.6999
val Loss: 0.6796 Acc: 0.7881
Epoch 4/10
----------
train Loss: 0.7977 Acc: 0.7118
val Loss: 0.7407 Acc: 0.7548
Epoch 5/10
----------
train Loss: 0.7883 Acc: 0.7137
val Loss: 0.6673 Acc: 0.7962
Epoch 6/10
----------
train Loss: 0.7728 Acc: 0.7144
val Loss: 0.6822 Acc: 0.7889
Epoch 7/10
----------
train Loss: 0.7699 Acc: 0.7156
val Loss: 0.6918 Acc: 0.7885
Epoch 8/10
----------
train Loss: 0.7562 Acc: 0.7230
val Loss: 0.7099 Acc: 0.7574
Epoch 9/10
----------
train Loss: 0.7406 Acc: 0.7292
val Loss: 0.7476 Acc: 0.7428
Epoch 10/10
----------
train Loss: 0.7328 Acc: 0.7311
val Loss: 0.7179 Acc: 0.7669
Best val Acc: 0.796195


In [21]:
print("QAT para pesos de 4 bits e ativações de 2 bits.")
model_name = 'qat_resnet18_w4_a2.pth'

quant_model = QuantResNet18(4,2)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 4 bits e ativações de 2 bits.
Epoch 1/10
----------


/tmp/ipykernel_189/3438579381.py:107: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x,1)


train Loss: 5.7942 Acc: 0.4342
val Loss: 1.3076 Acc: 0.5020
Epoch 2/10
----------
train Loss: 1.4313 Acc: 0.4578
val Loss: 1.1688 Acc: 0.5313
Epoch 3/10
----------
train Loss: 1.3069 Acc: 0.4869
val Loss: 1.1278 Acc: 0.5628
Epoch 4/10
----------
train Loss: 1.2688 Acc: 0.4933
val Loss: 1.1093 Acc: 0.5543
Epoch 5/10
----------
train Loss: 1.2436 Acc: 0.5064
val Loss: 1.0776 Acc: 0.5847
Epoch 6/10
----------
train Loss: 1.2085 Acc: 0.5233
val Loss: 1.0720 Acc: 0.5562
Epoch 7/10
----------
train Loss: 1.2153 Acc: 0.5203
val Loss: 1.0218 Acc: 0.5763
Epoch 8/10
----------
train Loss: 1.1854 Acc: 0.5326
val Loss: 1.0160 Acc: 0.5939
Epoch 9/10
----------
train Loss: 1.1718 Acc: 0.5422
val Loss: 1.0026 Acc: 0.6092
Epoch 10/10
----------
train Loss: 1.1610 Acc: 0.5453
val Loss: 0.9812 Acc: 0.6550
Best val Acc: 0.654958


# Avaliando os modelos quantizados

In [145]:
def calculate_metrics(loader, model, device):
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)], output_dict=True)
    kappa = cohen_kappa_score(all_labels, all_preds)
    
    per_class_accuracy = [report[str(i)]['precision'] for i in range(6)]
    mean_accuracy = sum(per_class_accuracy) / len(per_class_accuracy)

    return accuracy, mean_accuracy, kappa, classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)]), confusion_matrix(all_labels, all_preds)

In [146]:
def plot_confusion_matrix(cm, title, filename):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[str(i) for i in range(6)], yticklabels=[str(i) for i in range(6)])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.savefig(filename)
    plt.close()

## W8A8

In [147]:
quant_model=QuantResNet18(8,8)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_resnet18_w8_a8.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantResNet18(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [154]:
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w8a8 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w8a8 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w8a8 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet18_w8_a8.png')

Validation Set Metrics for w8a8 quantization:


/tmp/ipykernel_190/3438579381.py:107: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x,1)


Validation OA: 0.8203
Validation AA: 0.6689
Validation Kappa: 0.7193
Validation Classification Report for w8a8 quantization:
              precision    recall  f1-score   support

           0       0.84      0.96      0.90      1253
           1       0.19      0.04      0.07       126
           2       0.80      0.82      0.81       895
           3       0.35      0.34      0.34        47
           4       0.92      0.46      0.62       182
           5       0.92      0.88      0.90       230

    accuracy                           0.82      2733
   macro avg       0.67      0.58      0.60      2733
weighted avg       0.80      0.82      0.80      2733

Validation Confusion Matrix for w8a8 quantization:
[[1205   12   36    0    0    0]
 [  81    5   39    0    1    0]
 [ 125   10  730   19    3    8]
 [   0    0   28   16    3    0]
 [   7    0   70   11   84   10]
 [  19    0    9    0    0  202]]


In [155]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a8 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w8a8 quantization:")
print(test_report)
print("Test Confusion Matrix for w8a8 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet18_w8_a8.png')

Test Set Metrics for w8a8 quantization:
Test OA: 0.6850
Test AA: 0.5731
Test Kappa: 0.5018
Test Classification Report for w8a8 quantization:
              precision    recall  f1-score   support

           0       0.70      0.94      0.80      1880
           1       0.11      0.04      0.06       189
           2       0.65      0.45      0.53      1344
           3       0.33      0.18      0.23        71
           4       0.90      0.29      0.44       275
           5       0.75      0.97      0.84       346

    accuracy                           0.69      4105
   macro avg       0.57      0.48      0.49      4105
weighted avg       0.67      0.69      0.65      4105

Test Confusion Matrix for w8a8 quantization:
[[1772   22   85    0    0    1]
 [ 119    8   60    0    0    2]
 [ 633   36  602    8    3   62]
 [   4    4   49   13    1    0]
 [   5    0  121   19   81   49]
 [   0    0    5    0    5  336]]


## W8A4

In [156]:
quant_model=QuantResNet18(8,4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_resnet18_w8_a4.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantResNet18(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [157]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w8a4 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w8a4 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w8a4 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet18_w8_a4.png')

Validation Set Metrics for w8a4 quantization:
Validation OA: 0.7808
Validation AA: 0.6984
Validation Kappa: 0.6511
Validation Classification Report for w8a4 quantization:
              precision    recall  f1-score   support

           0       0.79      0.98      0.87      1253
           1       0.50      0.02      0.03       126
           2       0.75      0.72      0.73       895
           3       0.44      0.23      0.31        47
           4       0.79      0.49      0.60       182
           5       0.93      0.73      0.81       230

    accuracy                           0.78      2733
   macro avg       0.70      0.53      0.56      2733
weighted avg       0.77      0.78      0.75      2733

Validation Confusion Matrix for w8a4 quantization:
[[1223    0   30    0    0    0]
 [  88    2   35    0    1    0]
 [ 224    2  642   11   10    6]
 [   1    0   31   11    4    0]
 [  11    0   72    3   89    7]
 [   4    0   50    0    9  167]]


In [158]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a4 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w8a4 quantization:")
print(test_report)
print("Test Confusion Matrix for w8a4 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet18_w8_a4.png')

Test Set Metrics for w8a4 quantization:
Test OA: 0.6687
Test AA: 0.5635
Test Kappa: 0.4632
Test Classification Report for w8a4 quantization:
              precision    recall  f1-score   support

           0       0.66      0.98      0.79      1880
           1       0.00      0.00      0.00       189
           2       0.62      0.35      0.45      1344
           3       0.48      0.18      0.27        71
           4       0.83      0.38      0.52       275
           5       0.78      0.90      0.84       346

    accuracy                           0.67      4105
   macro avg       0.56      0.47      0.48      4105
weighted avg       0.64      0.67      0.62      4105

Test Confusion Matrix for w8a4 quantization:
[[1841    0   39    0    0    0]
 [ 128    0   60    0    0    1]
 [ 801    0  476    7    9   51]
 [  14    0   42   13    2    0]
 [   9    0  120    7  105   34]
 [   1    0   25    0   10  310]]


## W4A8

In [159]:
quant_model=QuantResNet18(4,8)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_resnet18_w4_a8.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantResNet18(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [160]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w4a8 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w4a8 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w4a8 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet18_w4_a8.png')

Validation Set Metrics for w4a8 quantization:
Validation OA: 0.8149
Validation AA: 0.6533
Validation Kappa: 0.7123
Validation Classification Report for w4a8 quantization:
              precision    recall  f1-score   support

           0       0.82      0.99      0.89      1253
           1       0.22      0.05      0.08       126
           2       0.83      0.72      0.77       895
           3       0.33      0.36      0.35        47
           4       0.81      0.66      0.73       182
           5       0.90      0.90      0.90       230

    accuracy                           0.81      2733
   macro avg       0.65      0.61      0.62      2733
weighted avg       0.79      0.81      0.80      2733

Validation Confusion Matrix for w4a8 quantization:
[[1235    3   15    0    0    0]
 [  95    6   24    0    1    0]
 [ 166   18  641   32   17   21]
 [   1    0   22   17    7    0]
 [   6    0   52    2  120    2]
 [   4    0   15    0    3  208]]


In [161]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a4 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w4a8 quantization:")
print(test_report)
print("Test Confusion Matrix for w4a8 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet18_w4_a8.png')

Test Set Metrics for w8a4 quantization:
Test OA: 0.6814
Test AA: 0.5965
Test Kappa: 0.4901
Test Classification Report for w4a8 quantization:
              precision    recall  f1-score   support

           0       0.66      0.99      0.79      1880
           1       0.14      0.01      0.02       189
           2       0.70      0.33      0.44      1344
           3       0.49      0.25      0.33        71
           4       0.84      0.57      0.68       275
           5       0.74      0.94      0.83       346

    accuracy                           0.68      4105
   macro avg       0.60      0.52      0.52      4105
weighted avg       0.67      0.68      0.63      4105

Test Confusion Matrix for w4a8 quantization:
[[1857    3   19    0    0    1]
 [ 135    2   49    0    0    3]
 [ 795    9  437   15   13   75]
 [  10    0   38   18    5    0]
 [   9    0   69    4  158   35]
 [   0    0   10    0   11  325]]


## W4A4

In [162]:
quant_model=QuantResNet18(4,4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_resnet18_w4_a4.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantResNet18(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [163]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w4a4 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w4a4 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w4a4 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_resnet18_w4_a4.png')

Validation Set Metrics for w4a4 quantization:
Validation OA: 0.7962
Validation AA: 0.6208
Validation Kappa: 0.6825
Validation Classification Report for w4a4 quantization:
              precision    recall  f1-score   support

           0       0.84      0.94      0.89      1253
           1       0.14      0.02      0.04       126
           2       0.75      0.79      0.77       895
           3       0.27      0.30      0.28        47
           4       0.82      0.46      0.59       182
           5       0.91      0.83      0.87       230

    accuracy                           0.80      2733
   macro avg       0.62      0.56      0.57      2733
weighted avg       0.77      0.80      0.78      2733

Validation Confusion Matrix for w4a4 quantization:
[[1177    5   71    0    0    0]
 [  78    3   43    0    1    1]
 [ 138   13  706   25   10    3]
 [   1    0   27   14    5    0]
 [   6    0   64   13   84   15]
 [   2    0   33    0    3  192]]


In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w4a4 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w4a4 quantization:")
print(test_report)
print("Test Confusion Matrix for w4a4 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_resnet18_w4_a4.png')

Test Set Metrics for w4a4 quantization:


# Exportando o modelo w4a4 para ONNX

In [7]:
quant_model=QuantResNet18(4,4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_resnet18_w4_a4.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantResNet18(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [8]:
quant_model.cpu()
quant_model.eval()

QuantResNet18(
  (conv1): QuantConv2d(
    3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
    (input_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (output_quant): ActQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
    )
    (weight_quant): WeightQuantProxyFromInjector(
      (_zero_hw_sentinel): StatelessBuffer()
      (tensor_quant): RescalingIntQuant(
        (int_quant): IntQuant(
          (float_to_int_impl): RoundSte()
          (tensor_clamp_impl): TensorClampSte()
          (delay_wrapper): DelayWrapper(
            (delay_impl): _NoDelay()
          )
        )
        (scaling_impl): StatsFromParameterScaling(
          (parameter_list_stats): _ParameterListStats(
            (first_tracked_param): _ViewParameterWrapper(
              (view_shape_impl): OverOutputChannelView(
                (permute_impl): Identity()
              )
            )
            (stats): _Stats(
              

In [9]:
quant_model_export = quant_model

In [10]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
ready_model_filename = model_dir + "/classifier-resnet18-w4a4-ready.onnx"

In [11]:
input_shape = (1,3,224,224)
input_a = np.random.randn(*input_shape).astype(np.float32)
input_t = torch.from_numpy(input_a)

export_qonnx(quant_model_export,export_path=ready_model_filename,input_t=input_t) 
qonnx_cleanup(ready_model_filename,out_file=ready_model_filename)

In [12]:
wrapped_model = ModelWrapper(ready_model_filename)
wrapped_model.set_tensor_datatype(wrapped_model.graph.input[0].name,DataType["FLOAT32"])
wrapped_model = wrapped_model.transform(ConvertQONNXtoFINN())

/home/isadora/Xilinx/finn/deps/qonnx/src/qonnx/transformation/gemm_to_matmul.py:57: UserWarning: The GemmToMatMul transformation only offers explicit support for version 9 of the Gemm node, but the ONNX version of the supplied model is 14. Thus the transformation may fail or return incomplete results.
  warnings.warn(


In [13]:
wrapped_model.save(ready_model_filename)
print(f"Model saved to {ready_model_filename}")

Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet18-w4a4-ready.onnx


In [14]:
finnonnx_in_tensor_name = wrapped_model.graph.input[0].name
finnonnx_out_tensor_name = wrapped_model.graph.output[0].name
print("Input tensor name: %s" % finnonnx_in_tensor_name)
print("Output tensor name: %s" % finnonnx_out_tensor_name)
finnonnx_model_in_shape = wrapped_model.get_tensor_shape(finnonnx_in_tensor_name)
finnonnx_model_out_shape = wrapped_model.get_tensor_shape(finnonnx_out_tensor_name)
print("Input tensor shape: %s" % str(finnonnx_model_in_shape))
print("Output tensor shape: %s" % str(finnonnx_model_out_shape))
finnonnx_model_in_dt = wrapped_model.get_tensor_datatype(finnonnx_in_tensor_name)
finnonnx_model_out_dt = wrapped_model.get_tensor_datatype(finnonnx_out_tensor_name)
print("Input tensor datatype: %s" % str(finnonnx_model_in_dt.name))
print("Output tensor datatype: %s" % str(finnonnx_model_out_dt.name))
print("List of node operator types in the graph: ")
print([x.op_type for x in wrapped_model.graph.node])

Input tensor name: global_in
Output tensor name: global_out
Input tensor shape: [1, 3, 224, 224]
Output tensor shape: [1, 6]
Input tensor datatype: FLOAT32
Output tensor datatype: FLOAT32
List of node operator types in the graph: 
['Conv', 'Mul', 'BatchNormalization', 'MultiThreshold', 'Mul', 'MaxPool', 'Conv', 'Mul', 'BatchNormalization', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'BatchNormalization', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'BatchNormalization', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'BatchNormalization', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Conv', 'Mul', 'Mul', 'BatchNormalization', 'BatchNormalization', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'BatchNormalization', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'BatchNormalization', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'BatchNormalization', 'Add', 'MultiThreshold', 'Mul', 'Conv', 'Conv', 'Mul', 'Mul', 'BatchNormalization', 'BatchNormalization', 'MultiThreshold', 'Mul', 'Conv', 'Mul', 'BatchNormalization'

# Gerando resultados estimados

In [144]:
def step_resnet18_streamline(model: ModelWrapper, cfg: DataflowBuildConfig) -> ModelWrapper:
    streamline_transformations = [
        MoveMulPastFork(),
        MoveLinearPastEltwiseAdd(),
        ConvertDivToMul(),
        BatchNormToAffine(),
        MoveMulPastMaxPool(), # added Got it right
        MoveMulPastFork(),
        MoveTransposePastScalarMul(),
        MoveScalarMulPastConv(),
        CollapseRepeatedMul(),
        MoveAddPastConv(),
        CollapseRepeatedAdd(),
        AbsorbMulIntoMultiThreshold(),
        MoveAddPastMul(),
        CollapseRepeatedAdd(),
        CollapseRepeatedMul(),
        AbsorbAddIntoMultiThreshold(),
        FactorOutMulSignMagnitude(),
        AbsorbMulIntoMultiThreshold(),
        Absorb1BitMulIntoConv(),
        MoveScalarMulPastConv(),
        CollapseRepeatedMul(),
        AbsorbMulIntoMultiThreshold(),
        MoveIdenticalOpPastJoinOp(['Mul'], ['Add']), #added
        MoveAddPastConv(),
        AbsorbAddIntoMultiThreshold(),
        AbsorbMulIntoMultiThreshold(),
        AbsorbSignBiasIntoMultiThreshold(),
        MoveScalarLinearPastInvariants(),
        MoveScalarMulPastMatMul(),
        AbsorbMulIntoMultiThreshold(),
        CollapseRepeatedMul(),
        AbsorbScalarMulAddIntoTopK(),
        RemoveIdentityOps(),
    ]

    for t in streamline_transformations:
        model = model.transform(t)
    
    if VerificationStepType.STREAMLINED_PYTHON in cfg._resolve_verification_steps():
        verify_step(model, cfg, "streamlined_python", need_parent=False)

    return cleanup_model(model)

In [55]:
def step_resnet18_lower(model: ModelWrapper, cfg: DataflowBuildConfig) -> ModelWrapper:
    model = model.transform(LowerConvsToMatMul())
    model = model.transform(AbsorbTransposeIntoMultiThreshold())
    model = model.transform(AbsorbConsecutiveTransposes())
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(GiveReadableTensorNames())
    model = model.transform(InferDataTypes())
    model = model.transform(RoundAndClipThresholds())
    return cleanup_model(model)

In [143]:
def step_resnet18_to_hw(model: ModelWrapper, cfg: DataflowBuildConfig) -> ModelWrapper:
    to_hw_transforms = [
        RoundAndClipThresholds(),
        DoubleToSingleFloat(),
        InferDataTypes(),
        SortGraph(),
        InferShapes(),
        InferAddStreamsLayer(),
        InferPool(),
        AbsorbMulIntoMultiThreshold(),
        RoundAndClipThresholds(),
        InferThresholdingLayer(),
        InferQuantizedMatrixVectorActivation(),
        AbsorbConsecutiveTransposes(),
        InferConvInpGen(),
        InferDuplicateStreamsLayer(),
        InferAddStreamsLayer(),
        AbsorbConsecutiveTransposes(),
        AbsorbTransposeIntoFlatten(),
        AbsorbMulIntoMultiThreshold(),
        AbsorbTransposeIntoMultiThreshold(),
        MoveTransposePastFork(),
        AbsorbConsecutiveTransposes(),
        RemoveCNVtoFCFlatten()
    ]
    
    # Workaround for an error. If it's not included, the first Im2Col nod
    # is not converted to an (FMPadding_Batch -> ConvolutionInputGenerator)
    model.set_tensor_datatype(model.graph.input[0].name, DataType["UINT8"])
    
    for t in to_hls_transformations:
        model = model.transform(InferDataLayouts())
        model = model.transform(t)
        model = model.transform(GiveUniqueNodeNames())
        model = model.transform(InferDataTypes())

    model = model.transform(InferShapes())

    return cleanup_model(model)

In [141]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_ResNet18"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate = build_cfg.DataflowBuildConfig(
    steps=[
        "step_tidy_up",
        step_resnet18_streamline,
        step_resnet18_lower,
        step_resnet18_to_hw,
        "step_create_dataflow_partition",
        "step_specialize_layers",
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

Previous run results deleted!


In [142]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-resnet18-ready.onnx", estimate)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-resnet18-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet18
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_ResNet18/build_dataflow.log
Running step: step_tidy_up [1/10]
Running step: step_resnet18_streamline [2/10]
Running step: step_resnet18_lower [3/10]
Running step: step_resnet18_to_hw [4/10]
Running step: step_create_dataflow_partition [5/10]
Running step: step_specialize_layers [6/10]
Running step: step_target_fps_parallelization [7/10]
Running step: step_apply_folding_config [8/10]
Running step: step_minimize_bit_width [9/10]
Running step: step_generate_estimate_reports [10/10]
Completed successfully
CPU times: user 7.58 s, sys: 1.23 s, total: 8.81 s
Wall time: 8.8 s


0